# mboost — numerical parity with R

This notebook compares selected fits with R's `mboost` implementation. Agreement is evaluated against the tolerances listed below; the tolerances differ by method.

This notebook fits the same models in R (via rpy2) and mboost, then reports element-wise discrepancies across:

1. **Families** — Gaussian, Poisson, GammaReg, QuantReg, ExpectReg, Huber.
2. **Spline mechanics** — bbs(...) P-splines across a sweep of df, knots, differences.
3. **gamboost mixed terms** — bbs + bols risk path and per-iteration selected-learner sequence.
4. **Weighted fits** — non-uniform observation weights.
5. **Prediction** — glmboost on training rows, type='link' and type='response'.
6. **Sliced fits** — glmboost[:k] equivalence with R's fit[k].
7. **Cross-validation** — cvrisk with kfold, bootstrap, and subsampling.
8. **Monotone splines (bmono)** — increasing-constrained P-spline.
9. **Summary ledger** — every comparison with its tolerance bucket and a log-scale chart.

Tolerance budget:

| Component                     | Target relative tol |
|-------------------------------|---------------------|
| glmboost Gaussian / Poisson   | 1e-10               |
| Other glmboost families       | 1e-8                |
| gamboost with bbs(...)        | 1e-6                |
| bmono (iterative QP)          | 1e-4                |
| cvrisk (risk matrix / aggregate) | 1e-6 / 1e-10    |


In [ ]:
import numpy as np
import polars as pl
import altair as alt

import mboostrs as mb
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

r_mboost = importr('mboost')
r_base = importr('base')

print('mboost (Rust) :', mb.__version__)
print('mboost (R)    :', '.'.join(map(str, ro.r('packageVersion("mboost")')[0])))
print('R             :', ro.r('R.version.string')[0])

In [2]:
# Cumulative parity ledger — every check appends a row; summarised at the end.
LEDGER = []

def check(group, name, ours, rs, tol_rel=1e-8, tol_abs=1e-10):
    ours = np.asarray(ours, dtype=np.float64).ravel()
    rs   = np.asarray(rs,   dtype=np.float64).ravel()
    abs_diff = float(np.max(np.abs(ours - rs)))
    denom = float(np.max(np.abs(rs)))
    rel_diff = abs_diff / denom if denom > 0 else 0.0
    passed = (abs_diff <= tol_abs) or (rel_diff <= tol_rel)
    LEDGER.append((group, name, abs_diff, rel_diff, tol_rel, passed))
    marker = 'PASS' if passed else 'FAIL'
    print(f'  [{marker}] {name:<28s} max|Δ|={abs_diff:.2e}  rel={rel_diff:.2e}  (tol={tol_rel:.0e})')
    return passed

## 1. Family gallery — every supported glmboost family

Each family supplies (offset, negative_gradient, risk) to the engine. We fit y ~ x1 + x2 in both R and mboost with the same data and check every output.
R's names differ for two families: Quantile(\u03c4) -> mboost::QuantReg(tau=\u03c4) and Expectile(\u03c4) -> mboost::ExpectReg(tau=\u03c4).

Data generation matches each family (binary y for Binomial, y > 0 for GammaReg, etc.).

In [ ]:
def r_fit_family(family_r, dgp_r, n=200, mstop=80, nu=0.1, seed=42):
    """Generate data via `dgp_r` (R code producing x1, x2, y) and fit
    `glmboost(y ~ x1 + x2, family=<family_r>, center=TRUE)` in R."""
    src = f'''function(n, mstop, nu, seed) {{
        set.seed(seed)
        x1 <- rnorm(n); x2 <- rnorm(n)
        {dgp_r}
        fit <- mboost::glmboost(
            y ~ x1 + x2,
            data    = data.frame(x1=x1, x2=x2, y=y),
            family  = {family_r},
            center  = TRUE,
            control = mboost::boost_control(mstop=mstop, nu=nu))
        cc <- stats::coef(fit, off2int=FALSE)
        list(x1=as.numeric(x1), x2=as.numeric(x2), y_resp=as.numeric(y),
             fitted=as.numeric(stats::predict(fit, type="link")),
             risk=as.numeric(mboost::risk(fit)),
             coef_names=names(cc), coef=as.numeric(cc),
             offset=as.numeric(attr(cc, "offset")))
    }}'''
    r = ro.r(src)(n, mstop, nu, seed)
    return {
        'x1': np.asarray(r[0]), 'x2': np.asarray(r[1]), 'y': np.asarray(r[2]),
        'fitted': np.asarray(r[3]), 'risk': np.asarray(r[4]),
        'coef_names': list(r[5]), 'coef': np.asarray(r[6]),
        'offset': float(r[7][0]) if len(r[7]) else 0.0,
    }

FAMILIES = [
    ('Gaussian',       'mboost::Gaussian()',         'y <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.3)',                       mb.Gaussian(),         1e-10),
    ('Poisson',        'mboost::Poisson()',          'y <- rpois(n, lambda=exp(0.3*x1 + 0.1*x2))',                    mb.Poisson(),          1e-10),
    ('GammaReg',       'mboost::GammaReg()',         'y <- exp(0.3*x1 + 0.1*x2) * rgamma(n, shape=2, rate=2)',        mb.GammaReg(),         1e-8),
    ('QuantReg(0.50)', 'mboost::QuantReg(tau=0.50)', 'y <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.5)',                       mb.Quantile(tau=0.50), 1e-8),
    ('ExpectReg(0.5)', 'mboost::ExpectReg()',        'y <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.5)',                       mb.Expectile(tau=0.5), 1e-8),
    ('Huber',          'mboost::Huber()',            'y <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.5)',                       mb.Huber(),            1e-8),
]

for label, r_family, dgp, py_family, tol in FAMILIES:
    r = r_fit_family(r_family, dgp)
    data = pl.DataFrame({'y': r['y'], 'x1': r['x1'], 'x2': r['x2']})
    fit = mb.glmboost('y ~ x1 + x2', data=data, family=py_family,
                      control=mb.boost_control(mstop=80, nu=0.1))
    print(f'\n--- {label} ---')
    check(label, 'risk',   fit.risk_,   r['risk'],   tol_rel=tol)
    check(label, 'fitted', fit.fitted_, r['fitted'], tol_rel=tol)
    r_named = dict(zip(r['coef_names'], r['coef']))
    for nm in ('x1', 'x2'):
        if nm in r_named and nm in fit.coefficients_:
            check(label, f'coef[{nm}]', [fit.coefficients_[nm]], [r_named[nm]], tol_rel=tol)

## 2. bbs(...) P-spline sweep

bbs builds a B-spline basis, applies a difference penalty K = D'D, and converts effective degrees of freedom to ridge weight \u03bb via the Demmler-Reinsch identity.
We exercise it across:

- df: 3, 4, 8 (R default 4).
- knots: 10, 20, 30 (R default 20).
- degree: 3 (cubic; universal default).
- differences: 1, 2 (R default 2).

Each setting: y ~ bbs(x, df=..., knots=..., degree=..., differences=...) on a sinusoidal target. We compare risk path and fitted values.

In [ ]:
def r_fit_bbs(df, knots, degree, differences, mstop=80, nu=0.1, n=200, seed=42):
    src = f'''function() {{
        library(mboost)
        set.seed({seed})
        x1 <- runif({n}, 0, 1)
        y  <- sin(2*pi*x1) + rnorm({n}, sd=0.3)
        fit <- gamboost(
            y ~ bbs(x1, df={df}, knots={knots}, degree={degree}, differences={differences}),
            data = data.frame(x1=x1, y=y),
            family = Gaussian(),
            control = boost_control(mstop={mstop}, nu={nu}))
        list(x1=as.numeric(x1), y=as.numeric(y),
             fitted=as.numeric(stats::predict(fit, type="link")),
             risk=as.numeric(mboost::risk(fit)))
    }}'''
    r = ro.r(src)()
    return {'x1': np.asarray(r[0]), 'y': np.asarray(r[1]),
            'fitted': np.asarray(r[2]), 'risk': np.asarray(r[3])}

SWEEP = [
    dict(df=3, knots=20, degree=3, differences=2),
    dict(df=4, knots=20, degree=3, differences=2),
    dict(df=8, knots=20, degree=3, differences=2),
    dict(df=4, knots=10, degree=3, differences=2),
    dict(df=4, knots=30, degree=3, differences=2),
    dict(df=4, knots=20, degree=3, differences=1),
]
for cfg in SWEEP:
    r = r_fit_bbs(**cfg)
    data = pl.DataFrame({'y': r['y'], 'x1': r['x1']})
    formula = (f"y ~ bbs(x1, df={cfg['df']}, knots={cfg['knots']}, "
               f"degree={cfg['degree']}, differences={cfg['differences']})")
    fit = mb.gamboost(formula, data=data, control=mb.boost_control(80, 0.1))
    label = f"bbs[df={cfg['df']},knots={cfg['knots']},diff={cfg['differences']}]"
    print(f'\n--- {label} ---')
    check(label, 'risk',   fit.risk_,   r['risk'],   tol_rel=1e-6)
    check(label, 'fitted', fit.fitted_, r['fitted'], tol_rel=1e-6)

## 3. gamboost with mixed terms + selected-learner sequence

When the formula combines learners of different kinds (bbs(x1) + bols(x2)), R's tie-break rule picks the lowest-indexed learner.
We compare the selected-learner sequence at each iteration and the final risk.

In [ ]:
r = ro.r('''function() {
    library(mboost)
    set.seed(42); n <- 200
    x1 <- runif(n, 0, 1); x2 <- rnorm(n)
    y  <- sin(2*pi*x1) + 0.5*x2 + rnorm(n, sd=0.3)
    fit <- gamboost(y ~ bbs(x1, df=4, knots=20) + bols(x2),
                    data=data.frame(x1=x1, x2=x2, y=y),
                    family=Gaussian(),
                    control=boost_control(mstop=120, nu=0.1))
    list(x1=as.numeric(x1), x2=as.numeric(x2), y=as.numeric(y),
         fitted=as.numeric(stats::predict(fit, type="link")),
         risk=as.numeric(mboost::risk(fit)),
         sel=as.integer(mboost::selected(fit)))
}''')()
x1, x2, y = np.asarray(r[0]), np.asarray(r[1]), np.asarray(r[2])
r_fit, r_risk = np.asarray(r[3]), np.asarray(r[4])
r_sel = np.asarray(r[5], dtype=np.int32)

data = pl.DataFrame({'y': y, 'x1': x1, 'x2': x2})
fit = mb.gamboost('y ~ bbs(x1, df=4, knots=20) + bols(x2)', data=data,
                  control=mb.boost_control(120, 0.1))
our_sel = np.asarray(fit._selected, dtype=np.int32) + 1  # R is 1-indexed

print('gamboost mixed terms — parity:')
check('gamboost.mixed', 'risk',   fit.risk_,   r_risk, tol_rel=1e-6)
check('gamboost.mixed', 'fitted', fit.fitted_, r_fit,  tol_rel=1e-6)
if our_sel.shape == r_sel.shape:
    agree = int((our_sel == r_sel).sum())
else:
    agree = 0
n_iter = max(len(r_sel), len(our_sel))
print(f'  selected-learner sequence: {agree}/{n_iter} iterations agree exactly')
LEDGER.append(('gamboost.mixed', 'selected_seq', n_iter - agree, 0.0, 0.0, agree == n_iter))

import pandas as pd
sel_df = pd.DataFrame({
    "iteration": list(range(1, 81)) * 2,
    "learner": np.concatenate([r_sel[:80], our_sel[:80]]),
    "implementation": ["R mboost"] * 80 + ["mboostrs"] * 80,
})
sel_df["learner"] = sel_df["learner"].map({1: "bbs(x1)", 2: "bols(x2)"})
pts_r = alt.Chart(sel_df[sel_df.implementation == "R mboost"]).mark_circle(size=40, opacity=0.7).encode(
    x="iteration:Q", y="learner:N", color=alt.value("#1f77b4")
)
pts_our = alt.Chart(sel_df[sel_df.implementation == "mboostrs"]).mark_point(shape="cross", size=50, opacity=0.7).encode(
    x="iteration:Q", y="learner:N", color=alt.value("#d62728")
)
(pts_r + pts_our).properties(
    title="Selected learner at each step (first 80)",
    width=500, height=200
).resolve_scale(color="independent")

## 4. Weighted fits

Non-uniform observation weights enter the design Gram (X\u1d40WX), the gradient (\u03a3 w\u1d62 u\u1d62), and the df\u2192\u03bb solver.

Note: R's risk(fit) returns the weighted loss while mboost risk_ aggregates without folding weights into the path scalar — a normalization difference, not a fit difference. We compare only fitted values and coefficients.

In [ ]:
r = ro.r('''function() {
    set.seed(7); n <- 200
    x1 <- rnorm(n); x2 <- rnorm(n)
    y  <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.3)
    w  <- runif(n, 0.2, 2.0)
    fit <- mboost::glmboost(y~x1+x2,
        data=data.frame(x1=x1, x2=x2, y=y), weights=w,
        family=mboost::Gaussian(), center=TRUE,
        control=mboost::boost_control(mstop=80, nu=0.1))
    cc <- stats::coef(fit, off2int=FALSE)
    list(x1=as.numeric(x1), x2=as.numeric(x2), y=as.numeric(y), w=as.numeric(w),
         fitted=as.numeric(stats::predict(fit, type="link")),
         coef_names=names(cc), coef=as.numeric(cc))
}''')()
x1, x2, y, w = (np.asarray(r[i]) for i in range(4))
r_fit = np.asarray(r[4])
r_named = dict(zip(list(r[5]), np.asarray(r[6])))

data = pl.DataFrame({'y': y, 'x1': x1, 'x2': x2})
fit = mb.glmboost('y~x1+x2', data=data, family=mb.Gaussian(),
                  control=mb.boost_control(80, 0.1), weights=w)
print('Weighted glmboost — parity:')
check('weighted', 'fitted', fit.fitted_, r_fit, tol_rel=1e-10)
for nm in ('x1', 'x2'):
    check('weighted', f'coef[{nm}]', [fit.coefficients_[nm]], [r_named[nm]], tol_rel=1e-10)

## 5. Prediction

glmboost.predict(type='link'|'response') should reproduce R's predict.mboost on the training rows.

The R prediction methods are also compared on new data and sliced fits in the standalone parity suite (`tests/test_predict_parity/test_gamboost_predict.py`).

In [ ]:
# (a) glmboost predict link/response on training rows.
r = ro.r('''function() {
    set.seed(42); n <- 200
    x1 <- rnorm(n); x2 <- rnorm(n)
    y  <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.3)
    fit <- mboost::glmboost(y~x1+x2,
        data=data.frame(x1=x1, x2=x2, y=y),
        family=mboost::Gaussian(), center=TRUE,
        control=mboost::boost_control(mstop=50, nu=0.1))
    list(x1=as.numeric(x1), x2=as.numeric(x2), y=as.numeric(y),
         pred_link=as.numeric(stats::predict(fit, type="link")),
         pred_resp=as.numeric(stats::predict(fit, type="response")))
}''')()
x1, x2, y = np.asarray(r[0]), np.asarray(r[1]), np.asarray(r[2])
r_link, r_resp = np.asarray(r[3]), np.asarray(r[4])
data = pl.DataFrame({'y': y, 'x1': x1, 'x2': x2})
fit = mb.glmboost('y~x1+x2', data=data, family=mb.Gaussian(),
                  control=mb.boost_control(50, 0.1))
print('glmboost predict (training rows):')
check('predict.glm', 'link',     fit.predict(type='link'),     r_link, tol_rel=1e-10)
check('predict.glm', 'response', fit.predict(type='response'), r_resp, tol_rel=1e-10)

## 6. Sliced fit — model[:k] equals R's fit[k]

Trimming the boosting path to a smaller k is the workflow mstop(cvrisk(...)) users rely on. Both engines should produce the same fitted values and risk-path prefix at k.

In [ ]:
# glmboost slice — model[:k] should match R's fit[k]
r = ro.r('''function() {
    set.seed(42); n <- 200
    x1 <- rnorm(n); x2 <- rnorm(n)
    y  <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.3)
    fit <- mboost::glmboost(y~x1+x2,
        data=data.frame(x1=x1, x2=x2, y=y),
        family=mboost::Gaussian(), center=TRUE,
        control=mboost::boost_control(mstop=100, nu=0.1))
    fit_k <- fit[30]
    list(x1=as.numeric(x1), x2=as.numeric(x2), y=as.numeric(y),
         fitted_k=as.numeric(stats::predict(fit_k, type="link")),
         risk_k=as.numeric(mboost::risk(fit_k)))
}''')()
x1, x2, y = np.asarray(r[0]), np.asarray(r[1]), np.asarray(r[2])
r_fit_k, r_risk_k = np.asarray(r[3]), np.asarray(r[4])
data = pl.DataFrame({'y': y, 'x1': x1, 'x2': x2})
fit = mb.glmboost('y~x1+x2', data=data, control=mb.boost_control(100, 0.1))
fit_k = fit[:30]
print('glmboost slice fit[:30]:')
check('slice.glm', 'fitted', fit_k.fitted_, r_fit_k,  tol_rel=1e-10)
check('slice.glm', 'risk',   fit_k.risk_,   r_risk_k, tol_rel=1e-10)

## 7. Cross-validation parity (kfold, bootstrap, subsampling)

cvrisk returns a (B \u00d7 (mstop+1)) matrix of out-of-bag risks plus an aggregate column-sum path; mstop(cvr) is its argmin.
We pass R-generated fold matrices into both engines so only the per-fold risk evaluation is tested.

Six combinations: {glm, gam} \u00d7 {kfold, bootstrap, subsampling}.

In [ ]:
def r_cv(fold_type, B, model_kind, mstop=50, nu=0.1, n=200, seed=42, prob=0.5):
    if model_kind == 'glm':
        model_block = '''
            x1 <- rnorm(n); x2 <- rnorm(n)
            y  <- 1.5*x1 + 0.8*x2 + rnorm(n, sd=0.3)
            d  <- data.frame(x1=x1, x2=x2, y=y)
            fit <- mboost::glmboost(y~x1+x2, data=d, center=TRUE,
                                    control=mboost::boost_control(mstop=mstop, nu=nu))
        '''
        formula = 'y~x1+x2'
        build = lambda data, mstop, nu: mb.glmboost(formula, data=data,
                                                    control=mb.boost_control(mstop, nu))
    else:
        model_block = '''
            x1 <- runif(n, 0, 1); x2 <- rnorm(n)
            y  <- sin(2*pi*x1) + 0.5*x2 + rnorm(n, sd=0.3)
            d  <- data.frame(x1=x1, x2=x2, y=y)
            fit <- mboost::gamboost(y ~ bbs(x1, df=4, knots=20) + bols(x2),
                                    data=d,
                                    control=mboost::boost_control(mstop=mstop, nu=nu))
        '''
        formula = 'y ~ bbs(x1, df=4, knots=20) + bols(x2)'
        build = lambda data, mstop, nu: mb.gamboost(formula, data=data,
                                                    control=mb.boost_control(mstop, nu))

    cv_args = {'kfold':       f'type="kfold", B={B}',
               'bootstrap':   f'type="bootstrap", B={B}',
               'subsampling': f'type="subsampling", B={B}, prob={prob}'}[fold_type]
    src = f'''function(n, mstop, nu, seed) {{
        library(mboost)
        set.seed(seed)
        {model_block}
        folds <- mboost::cv(model.weights(fit), {cv_args})
        cvr <- mboost::cvrisk(fit, folds=folds, papply=lapply)
        list(x1=as.numeric(x1), x2=as.numeric(x2), y=as.numeric(y),
             folds=as.matrix(folds), risk_matrix=as.matrix(cvr),
             colsums=as.numeric(colSums(cvr, na.rm=TRUE)),
             mstop_opt=as.integer(mboost::mstop(cvr)))
    }}'''
    r = ro.r(src)(n, mstop, nu, seed)
    return (np.asarray(r[0]), np.asarray(r[1]), np.asarray(r[2]),
            np.asarray(r[3], dtype=np.float64),
            np.asarray(r[4], dtype=np.float64),
            np.asarray(r[5], dtype=np.float64),
            int(r[6][0]), build)

for kind in ('glm', 'gam'):
    for fold_type, B in [('kfold', 10), ('bootstrap', 25), ('subsampling', 25)]:
        x1, x2, y, folds, r_rm, r_cs, r_opt, build = r_cv(fold_type, B, kind)
        data = pl.DataFrame({'y': y, 'x1': x1, 'x2': x2})
        fit = build(data, 50, 0.1)
        cvr = fit.cvrisk(folds=folds)
        label = f'cvrisk.{kind}.{fold_type}(B={B})'
        tol = 1e-10 if kind == 'glm' else 1e-6
        print(f'\n--- {label} ---')
        check(label, 'risk_matrix', cvr.risk_matrix, r_rm, tol_rel=tol)
        check(label, 'aggregate',   cvr.aggregate,   r_cs, tol_rel=tol)
        ok = cvr.optimal_mstop == r_opt
        print(f'  [{("PASS" if ok else "FAIL")}] mstop_opt                   '
              f'ours={cvr.optimal_mstop}  R={r_opt}')
        LEDGER.append((label, 'mstop_opt', abs(cvr.optimal_mstop - r_opt), 0.0, 0.0, ok))

## 8. bmono — monotone-constrained P-spline

bmono solves an inequality-constrained QP at each boosting step (min \u00bd\u03b2'A\u03b2 \u2212 b'\u03b2  s.t. D\u03b2 \u2265 0).
Our default solver is R's type="iterative" violation-penalty fixed point (\u03bb\u2082 = 1e6, 10 iters): fast enough for the inner loop, with ~1e-4 fidelity to a strict primal-dual QP.

In [ ]:
r = ro.r('''function() {
    library(mboost)
    set.seed(11); n <- 300
    x <- runif(n, -2, 2)
    y <- log(1+exp(2*x)) + rnorm(n, sd=0.2)   # monotone-increasing target
    fit <- gamboost(y ~ bmono(x, constraint="increasing"),
        data=data.frame(x=x, y=y), family=Gaussian(),
        control=boost_control(mstop=100, nu=0.1))
    list(x=as.numeric(x), y=as.numeric(y),
         fitted=as.numeric(stats::predict(fit, type="link")),
         risk=as.numeric(mboost::risk(fit)))
}''')()
x, y = np.asarray(r[0]), np.asarray(r[1])
r_fit, r_risk = np.asarray(r[2]), np.asarray(r[3])
data = pl.DataFrame({'y': y, 'x': x})
fit = mb.gamboost('y ~ bmono(x, constraint="increasing")', data=data,
                  control=mb.boost_control(100, 0.1))
print('bmono increasing — parity (iterative QP):')
check('bmono', 'risk',   fit.risk_,   r_risk, tol_rel=1e-4)
check('bmono', 'fitted', fit.fitted_, r_fit,  tol_rel=1e-4)

import pandas as pd
order = np.argsort(x)
bdf = pd.DataFrame({
    "x": x[order],
    "R bmono": np.asarray(r_fit)[order],
    "mboostrs bmono": np.asarray(fit.fitted_)[order],
})
pts = alt.Chart(pd.DataFrame({"x": x, "y": y})).mark_circle(size=8, opacity=0.25, color="gray").encode(x="x:Q", y="y:Q")
lines = alt.Chart(bdf).transform_fold(
    ["R bmono", "mboostrs bmono"], as_=["implementation", "fitted"]
).mark_line().encode(
    x="x:Q",
    y="fitted:Q",
    color="implementation:N",
    strokeDash="implementation:N",
)
(pts + lines).properties(title="Monotone P-spline \u2014 R vs. Rust", width=500, height=240)

## 9. Summary ledger

Each `check(...)` call records its comparison and tolerance in `LEDGER`. The final table reports the checks, and the log-scale bar chart compares each relative error with its tolerance.

In [ ]:
n_pass = sum(1 for *_, p in LEDGER if p)
n_tot  = len(LEDGER)
print(f'PARITY LEDGER  ({n_pass}/{n_tot} checks pass)\n')

import pandas as pd
ledger_df = pd.DataFrame([
    {"group": g, "check": n, "max_abs_diff": f"{a:.2e}", "rel_error": f"{r:.2e}", "status": "PASS" if p else "FAIL"}
    for g, n, a, r, _, p in LEDGER
])
display(ledger_df)

chart_df = pd.DataFrame([
    {"check": f"{g} \u00b7 {n}", "rel_error": max(r, 1e-18), "pass": p}
    for g, n, _, r, _, p in LEDGER
])
bars = alt.Chart(chart_df).mark_bar().encode(
    y=alt.Y("check:N", title=None, sort="-x", axis=alt.Axis(labels=False, ticks=False)),
    x=alt.X("rel_error:Q", title="relative error vs R (log scale)", scale=alt.Scale(type="log")),
    color=alt.Color("pass:N", scale=alt.Scale(domain=[True, False], range=["#2ca02c", "#d62728"]), legend=None),
    tooltip=["check:N", "rel_error:Q"],
).properties(title="Parity vs R \u2014 per-check relative error", width=600, height=max(300, 15 * n_tot))

# Tolerance reference lines
rules = alt.Chart(pd.DataFrame({"tol": [1e-10, 1e-8, 1e-6, 1e-4]})).mark_rule(
    strokeWidth=0.5, strokeDash=[3, 3], opacity=0.3
).encode(x="tol:Q")
labels = alt.Chart(pd.DataFrame({"tol": [1e-10, 1e-8, 1e-6, 1e-4], "label": ["1e-10", "1e-8", "1e-6", "1e-4"]})).mark_text(
    dy=-6, fontSize=9, opacity=0.5
).encode(x="tol:Q", text="label:N")

(bars + rules + labels)


---

For the **GAMLSS** distributional boosting story (`glmboostLSS` and `gamboostLSS`
with per-parameter formulas, cyclic/non-cyclic fitting, LSS cvrisk and stability
selection), see notebook `08_gamboostlss.ipynb`. Kernel, tensor, and `blackboost`
learners are tracked in `mb.capabilities()` as `planned` or `unsupported`.

**Next:** `03_performance.ipynb` times these same models, and states what each measurement's boundary is.
